# Notebook 12: Artificial Neural Network for Tabular Data

In this final notebook of the section, we will put everything together. We will take a real tabular dataset, preprocess it, and build an ANN to solve a classification problem. Finally, we will compare the ANN against a traditional ML model.

**Learning Methodology:**
**End-to-End Pipeline $\rightarrow$ Evaluation $\rightarrow$ Model Comparison $\rightarrow$ Interpretation**

## 1. Data Loading and Preprocessing

We will use a synthetic tabular dataset representing a classification problem (e.g., Customer Churn or Loan Default).

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load/Create Data
X, y = make_classification(n_samples=2000, n_features=15, random_state=42)
feature_names = [f"Feature_{i+1}" for i in range(15)]
df = pd.DataFrame(X, columns=feature_names)
df['Target'] = y

# 2. Train/Test Split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df.drop('Target', axis=1), df['Target'], test_size=0.2, random_state=42
)

# 3. Scaling (CRITICAL for ANN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# 4. Tensor Conversion
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

## 2. ANN Architecture and Training

We will build a network with:
Input (15) $\rightarrow$ Hidden (16) $\rightarrow$ ReLU $\rightarrow$ Hidden (8) $\rightarrow$ ReLU $\rightarrow$ Output (1) $\rightarrow$ Sigmoid.

In [ ]:
class TabularANN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(15, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = TabularANN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop
epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X_train_t)
    loss = criterion(preds, y_train_t)
    loss.backward()
    optimizer.step()

print("Training Complete.")

## 3. Evaluation

We will evaluate the model on the test set using multiple metrics.

In [ ]:
model.eval()
with torch.no_grad():
    test_preds_prob = model(X_test_t)
    test_preds = (test_preds_prob > 0.5).float()

# Metrics
acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds)
rec = recall_score(y_test, test_preds)
f1 = f1_score(y_test, test_preds)
auc = roc_auc_score(y_test, test_preds_prob)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

## 4. ANN vs Logistic Regression

Now we compare our Deep Learning model against a traditional baseline.

In [ ]:
# Logistic Regression Baseline
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_acc = accuracy_score(y_test, lr_preds)
print(f"ANN Accuracy: {acc:.4f}")
print(f"Logistic Regression Accuracy: {lr_acc:.4f}")

## 5. Final Analysis

**Which performed better?**
(In this synthetic example, the ANN usually wins if the patterns are non-linear).

**Comparison Table:**
| Feature | Logistic Regression | ANN |
| :--- | :--- | :--- |
| **Complexity** | Low (Linear) | High (Non-linear) |
| **Training Time** | Seconds | Minutes |
| **Interpretability** | High (Coefficients) | Low (Black Box) |
| **Data Req.** | Works on small data | Needs more data to shine |
| **Resources** | CPU | GPU (for large models) |